# Personal Finance & Spending Analysis
## 02 — Data Cleaning

**Objective:** Prepare a consistent dataset for analysis while preserving the original source.

**Input:** `data/raw_data.csv`  
**Outputs:** `data/cleaned_data.csv`, `reports/data_cleaning_summary.csv`

### Cleaning principles
- Never overwrite the raw dataset.
- Remove exact duplicate rows.
- Standardize column names and text categories.
- Convert expected numeric fields safely.
- Flag suspicious values and document changes.
- Retain missing values rather than applying unsupported imputation.

**Folder setup:** Save this notebook in `notebooks/`. The default paths below expect `raw_data.csv` in the project's `data/` folder.

## 1. Import Libraries and Configure Paths

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path("..")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw_data.csv"
CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "cleaned_data.csv"
REPORT_PATH = PROJECT_ROOT / "reports" / "data_cleaning_summary.csv"

CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Libraries imported and paths configured.")

: 

## 2. Load Raw Dataset

In [ ]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {RAW_DATA_PATH.resolve()}\n"
        "Place raw_data.csv in the project's data/ folder, "
        "or update RAW_DATA_PATH."
    )

df_raw = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded {df_raw.shape[0]:,} rows and {df_raw.shape[1]:,} columns.")
display(df_raw.head())

## 3. Create a Working Copy

In [ ]:
# Keep the original DataFrame unchanged.
df = df_raw.copy()
original_columns = df.columns.tolist()
print("Working copy created. Raw data remains unchanged.")

## 4. Standardize Column Names

In [ ]:
df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True)
      .str.strip("_")
)

duplicate_names = df.columns[df.columns.duplicated()].tolist()
if duplicate_names:
    raise ValueError(f"Duplicate column names after standardization: {duplicate_names}")

column_audit = pd.DataFrame({
    "original_column": original_columns,
    "standardized_column": df.columns
})
display(column_audit)

## 5. Remove Exact Duplicate Rows

In [ ]:
rows_before = len(df)
duplicate_rows_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

print(f"Rows before: {rows_before:,}")
print(f"Exact duplicates removed: {duplicate_rows_removed:,}")
print(f"Rows after: {len(df):,}")

## 6. Clean Text and Categorical Values

In [ ]:
text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

for column in text_columns:
    df[column] = df[column].astype("string").str.strip()
    df[column] = df[column].replace("", pd.NA)

# Normalize common categorical labels, if present.
for column in ["occupation", "city_tier"]:
    if column in df.columns:
        df[column] = (
            df[column].str.replace(r"\s+", "_", regex=True).str.lower()
        )
        print(f"\n{column} categories:")
        display(df[column].value_counts(dropna=False).to_frame("record_count"))

print("Text fields standardized.")

## 7. Convert Expected Numeric Columns

In [ ]:
expected_numeric_columns = [
    "income", "age", "dependents", "rent", "loan_repayment", "insurance",
    "groceries", "transport", "eating_out", "entertainment", "utilities",
    "healthcare", "education", "miscellaneous",
    "desired_savings_percentage", "desired_savings", "disposable_income",
    "potential_savings_groceries", "potential_savings_transport",
    "potential_savings_eating_out", "potential_savings_entertainment",
    "potential_savings_utilities", "potential_savings_healthcare",
    "potential_savings_education", "potential_savings_miscellaneous"
]

conversion_log = []
for column in expected_numeric_columns:
    if column in df.columns:
        converted = pd.to_numeric(df[column], errors="coerce")
        newly_missing = int((df[column].notna() & converted.isna()).sum())
        df[column] = converted
        conversion_log.append({
            "column": column,
            "values_coerced_to_missing": newly_missing
        })

display(pd.DataFrame(conversion_log))

## 8. Identify Invalid and Suspicious Values
Negative financial amounts are treated as invalid for this dataset and set to missing. Age and dependent-count issues are flagged for review rather than silently deleting records.

In [ ]:
non_negative_columns = [
    "income", "rent", "loan_repayment", "insurance", "groceries",
    "transport", "eating_out", "entertainment", "utilities",
    "healthcare", "education", "miscellaneous", "desired_savings",
    "disposable_income", "potential_savings_groceries",
    "potential_savings_transport", "potential_savings_eating_out",
    "potential_savings_entertainment", "potential_savings_utilities",
    "potential_savings_healthcare", "potential_savings_education",
    "potential_savings_miscellaneous"
]

negative_log = []
for column in non_negative_columns:
    if column in df.columns:
        mask = df[column] < 0
        count = int(mask.sum())
        negative_log.append({"column": column, "negative_values_found": count})
        df.loc[mask, column] = np.nan

display(pd.DataFrame(negative_log))

In [ ]:
# Savings percentage is expected to be between 0 and 100.
if "desired_savings_percentage" in df.columns:
    invalid_pct = (
        (df["desired_savings_percentage"] < 0) |
        (df["desired_savings_percentage"] > 100)
    )
    print("Out-of-range savings percentages:", int(invalid_pct.sum()))
    df.loc[invalid_pct, "desired_savings_percentage"] = np.nan

# Flag implausible values; do not automatically delete them.
if "age" in df.columns:
    df["age_review_flag"] = df["age"].notna() & (
        (df["age"] < 0) | (df["age"] > 100)
    )

if "dependents" in df.columns:
    df["dependents_review_flag"] = df["dependents"].notna() & (
        (df["dependents"] < 0) | (df["dependents"] % 1 != 0)
    )

for flag in ["age_review_flag", "dependents_review_flag"]:
    if flag in df.columns:
        print(f"{flag}: {int(df[flag].sum())} records flagged")

## 9. Review Missing Values

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100
}).sort_values("missing_percentage", ascending=False)

display(missing_summary)
print("Missing values are retained. No blanket imputation has been applied.")

## 10. Validate the Cleaned Dataset

In [ ]:
validation = {
    "rows_after_cleaning": len(df),
    "columns_after_cleaning": df.shape[1],
    "remaining_exact_duplicates": int(df.duplicated().sum()),
    "missing_cells": int(df.isna().sum().sum()),
    "negative_values_remaining_in_financial_fields": int(
        sum((df[c] < 0).sum() for c in non_negative_columns if c in df.columns)
    )
}
display(pd.DataFrame(list(validation.items()), columns=["check", "result"]))

In [ ]:
expected_core_columns = [
    "income", "age", "dependents", "occupation", "city_tier",
    "rent", "groceries", "transport", "desired_savings",
    "disposable_income"
]
core_check = pd.DataFrame({
    "column": expected_core_columns,
    "present": [c in df.columns for c in expected_core_columns]
})
display(core_check)

missing_core = core_check.loc[~core_check["present"], "column"].tolist()
if missing_core:
    print("Review missing expected columns:", missing_core)
else:
    print("All expected core columns are present.")

## 11. Compare Before and After Cleaning

In [ ]:
comparison = pd.DataFrame({
    "metric": ["Rows", "Columns", "Missing cells", "Exact duplicate rows"],
    "before_cleaning": [
        df_raw.shape[0], df_raw.shape[1],
        int(df_raw.isna().sum().sum()), int(df_raw.duplicated().sum())
    ],
    "after_cleaning": [
        df.shape[0], df.shape[1],
        int(df.isna().sum().sum()), int(df.duplicated().sum())
    ]
})
display(comparison)

## 12. Save Cleaned Dataset and Cleaning Report

In [ ]:
# Save to a new file. The raw CSV is never overwritten.
df.to_csv(CLEAN_DATA_PATH, index=False)

report_items = [
    ("raw_row_count", len(df_raw)),
    ("cleaned_row_count", len(df)),
    ("raw_column_count", df_raw.shape[1]),
    ("cleaned_column_count", df.shape[1]),
    ("exact_duplicate_rows_removed", duplicate_rows_removed),
    ("missing_cells_before", int(df_raw.isna().sum().sum())),
    ("missing_cells_after", int(df.isna().sum().sum())),
    ("remaining_exact_duplicates", int(df.duplicated().sum()))
]

for item in negative_log:
    report_items.append((f"negative_values_{item['column']}", item["negative_values_found"]))

cleaning_report = pd.DataFrame(report_items, columns=["check", "result"])
cleaning_report.to_csv(REPORT_PATH, index=False)

print(f"Cleaned data saved: {CLEAN_DATA_PATH.resolve()}")
print(f"Cleaning report saved: {REPORT_PATH.resolve()}")
display(cleaning_report)

## 13. Final Review

In [ ]:
print("DATA CLEANING SUMMARY")
print("-" * 45)
print(f"Raw rows:              {len(df_raw):,}")
print(f"Cleaned rows:          {len(df):,}")
print(f"Rows removed:          {len(df_raw) - len(df):,}")
print(f"Cleaned columns:       {df.shape[1]:,}")
print(f"Remaining duplicates:  {int(df.duplicated().sum()):,}")
print(f"Missing cells:         {int(df.isna().sum().sum()):,}")

display(df.head())